## Local Inference on GPU 
Model page: https://huggingface.co/microsoft/UserLM-8b

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/microsoft/UserLM-8b)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
%pip install -U transformers accelerate


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "microsoft/UserLM-8b"
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
).eval()

messages = [
    {
        "role": "system",
        "content": "You are a user asking for help implementing a sequence where each term is the sum of the two previous terms plus 1. The first two terms are 1 and 1.",
    }
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)
input_ids = inputs["input_ids"]
end_token_id = tokenizer.encode("<|eot_id|>", add_special_tokens=False)
end_conv_token_id = tokenizer.encode("<|endconversation|>", add_special_tokens=False)

outputs = model.generate(
    **inputs,
    do_sample=True,
    top_p=0.8,
    temperature=1.0,
    max_new_tokens=32,
    eos_token_id=end_token_id,
    pad_token_id=tokenizer.eos_token_id,
    bad_words_ids=[[token_id] for token_id in end_conv_token_id],
)
response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
print(response)
model_parameter_devices = sorted({str(p.device) for p in model.parameters()})
print("model parameter devices =", model_parameter_devices)
